In [83]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

In [84]:
from numpy.lib.stride_tricks import as_strided

In [85]:
x = np.arange(16).reshape((4,4))
x

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [12, 13, 14, 15]])

In [86]:
y = as_strided(x, shape=(2,     2,     3,   3), strides=(4*s, 1*s, 4*s, 1*s), check_bounds=True)
#                        out_h, out_w, k_h, k_w

In [87]:
def patches(X, k, stride):
    x_h, x_w = X.shape[:2]
    st_row, st_col = X.strides[:2]

    o_h = (x_h - k) // stride + 1
    o_w = (x_w - k) // stride + 1
    
    return as_strided(X, shape=(o_h, o_w, k, k), strides=(st_row*stride, st_col*stride, st_row, st_col), check_bounds=True)

In [ ]:
def conv(X, k, stride):
    p = patches(X, k.shape[0], stride)
    return np.tensordot(p,k)

In [88]:
k = np.arange(9).reshape((3,3))

In [ ]:
def avg_pool(X, k, stride):
    p = patches(X, k, stride)
    return np.mean(p, axis=(2,3))

array([[ 5.,  6.],
       [ 9., 10.]])

In [119]:
def pre_fc_forward(X, w1, w2):
    c1 = avg_pool(np.tanh(conv(X,w1,1)), 2, 2)
    return avg_pool(np.tanh(conv(c1,w2,1)), 2, 2)

In [ ]:
a_pool_d = lambda x_patches, k: 1/k**2 * np.einsum('ijkl->kl', x_patches, optimize='greedy')
tanh_d = lambda x: 1 - np.tanh(x)

In [122]:
def conv_d(x, dy, k, stride):
    p = patches(x, k.shape[0], stride)
    return np.tensordot(dy, p, axes=((0, 1), (0, 1)))

In [ ]:
def avgpool_d(dy,k):
    return np.repeat(np.repeat(dy / k**2, k, axis = 0),k , axis= 1)